# Coculture Treated - Mechanical Automatic Modeling
Decode processed data, fit coculture treated model zoo, and run sensitivity/uncertainty analysis.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

In [ ]:
using CSV, DataFrames, Plots, Dates, Statistics
include(joinpath(@__DIR__, "..", "src", "MechanicalAutomaticModeling.jl"))
using .MechanicalAutomaticModeling
using GrowthParameterEstimation

In [ ]:
condition = "coculture_treated"
decoded = MechanicalAutomaticModeling.IOUtils.decode_condition_dataframe(condition; start=@__DIR__)
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
decoded_path = joinpath(out.csv, "$(condition)_automatic_decoded.csv")
CSV.write(decoded_path, decoded)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="decode", outputs=[decoded_path], start=@__DIR__)
Text(repr(first(decoded, min(10, nrow(decoded)))))

In [ ]:
# Visualize decoded observations and their time-wise mean trend
obs_plot = scatter(
    decoded.time,
    decoded.count;
    alpha = 0.35,
    ms = 3,
    color = :steelblue,
    label = "observations",
    xlabel = "Time (day)",
    ylabel = "Cell count",
    title = "$(condition): observed decoded data",
    legend = :topleft,
)
time_means = combine(groupby(decoded, :time), :count => mean => :mean_count)
plot!(obs_plot, time_means.time, time_means.mean_count; lw = 3, color = :black, label = "mean by time")
display(obs_plot)

In [ ]:
fit_artifacts = MechanicalAutomaticModeling.FitWorkflows.run_condition_fit!(decoded, condition; start=@__DIR__)
Text(repr(first(fit_artifacts.ranking, min(10, nrow(fit_artifacts.ranking)))))

In [ ]:
# Visualize model quality (BIC) and model-vs-data overlay, when available
ranking_sorted = sort(fit_artifacts.ranking, :bic)
top_n = min(8, nrow(ranking_sorted))
bic_plot = bar(
    string.(ranking_sorted.model[1:top_n]),
    ranking_sorted.bic[1:top_n];
    legend = false,
    xlabel = "Model",
    ylabel = "BIC",
    title = "$(condition): top models by BIC",
    xrotation = 20,
)
display(bic_plot)

overlay_dir = joinpath(out.csv, "figures")
overlay_csvs = isdir(overlay_dir) ? sort(filter(f -> endswith(lowercase(f), "_overlay.csv"), readdir(overlay_dir; join = true))) : String[]

if isempty(overlay_csvs)
    println("No overlay CSV found in $(overlay_dir).")
else
    overlay = CSV.read(first(overlay_csvs), DataFrame)
    if !(:time in names(overlay) && :observed in names(overlay))
        println("Overlay CSV is missing required columns: time, observed")
    else
        p = scatter(
            overlay.time,
            overlay.observed;
            alpha = 0.4,
            ms = 3,
            color = :black,
            label = "observed",
            xlabel = "Time (day)",
            ylabel = "Cell count",
            title = "$(condition): model overlays",
        )

        pred_cols = filter(c -> startswith(String(c), "pred_"), names(overlay))
        plotted = 0
        for c in pred_cols
            mask = [!(ismissing(v) || (v isa AbstractFloat && isnan(v))) for v in overlay[!, c]]
            if any(mask)
                plot!(p, overlay.time[mask], overlay[mask, c]; lw = 2, label = replace(String(c), "pred_" => ""))
                plotted += 1
            end
        end

        if plotted == 0
            println("No non-missing model predictions found in overlay CSV.")
        end

        display(p)
    end
end

In [ ]:
analysis_artifacts = MechanicalAutomaticModeling.AnalysisWorkflows.run_condition_analysis!(decoded, fit_artifacts, condition; start=@__DIR__)
Text(repr(analysis_artifacts.sensitivity))

In [ ]:
out = MechanicalAutomaticModeling.IOUtils.condition_output_dirs(condition; start=@__DIR__)
summary = DataFrame(
    condition = [condition],
    decoded_rows = [nrow(decoded)],
    fit_rows = [nrow(fit_artifacts.ranking)],
    sensitivity_rows = [nrow(analysis_artifacts.sensitivity)],
    generated_at = [Dates.format(now(UTC), dateformat"yyyy-mm-ddTHH:MM:SS")]
)
summary_path = joinpath(out.metrics, "$(condition)_automatic_summary.csv")
CSV.write(summary_path, summary)
MechanicalAutomaticModeling.IOUtils.write_manifest_row(condition=condition, step="summary", outputs=[summary_path], start=@__DIR__)
Text(repr(summary))

## Final Plot: Best Fitted Model Over Data
This final cell overlays the best-BIC fitted model trajectory on top of the observed data and saves the figure to the condition image output folder.

In [ ]:
# Final overlay plot: best fitted model over observed data
ranking_sorted = sort(fit_artifacts.ranking, :bic)
best_model = String(ranking_sorted.model[1])
best_model_col_name = "pred_" * lowercase(best_model)

overlay_dir = joinpath(out.csv, "figures")
overlay_csvs = isdir(overlay_dir) ? sort(filter(f -> endswith(lowercase(f), "_overlay.csv"), readdir(overlay_dir; join = true))) : String[]

find_col(cols, candidates) = begin
    cand = Set(candidates)
    for c in cols
        lowercase(String(c)) in cand && return c
    end
    return nothing
end
finite_mask(vec) = [!(ismissing(v) || (v isa AbstractFloat && isnan(v))) for v in vec]

overlay = nothing
time_col = nothing
obs_col = nothing
pred_col = nothing
overlay_path = ""
selected_label = ""

for f in overlay_csvs
    df = CSV.read(f, DataFrame)
    tcol = find_col(names(df), ["time", "day", "days", "t"])
    ocol = find_col(names(df), ["observed", "count", "cell_count", "cells", "mean_count"])
    (tcol === nothing || ocol === nothing) && continue

    # First choice: best model prediction with finite values
    bcol = find_col(names(df), [best_model_col_name])
    if bcol !== nothing
        bmask = finite_mask(df[!, bcol])
        if any(bmask)
            overlay = df
            time_col = tcol
            obs_col = ocol
            pred_col = bcol
            overlay_path = f
            selected_label = "best model: $(best_model)"
            break
        end
    end

    # Fallback: any prediction column with finite values
    pred_candidates = filter(c -> startswith(lowercase(String(c)), "pred_"), names(df))
    for c in pred_candidates
        cmask = finite_mask(df[!, c])
        if any(cmask)
            overlay = df
            time_col = tcol
            obs_col = ocol
            pred_col = c
            overlay_path = f
            selected_label = "fallback: " * replace(String(c), "pred_" => "")
            break
        end
    end
    overlay !== nothing && break
end

if overlay === nothing
    println("No finite fitted predictions found in overlay CSVs; plotting observed data only.")
    best_fit_plot = scatter(
        decoded.time,
        decoded.count;
        alpha = 0.35,
        ms = 3,
        color = :black,
        label = "observed",
        xlabel = "Time (day)",
        ylabel = "Cell count",
        title = "$(condition): fitted model over data (prediction unavailable)",
        legend = :topleft,
    )
    time_means = combine(groupby(decoded, :time), :count => mean => :mean_count)
    plot!(best_fit_plot, time_means.time, time_means.mean_count; lw = 3, color = :gray30, label = "mean by time")
    best_overlay_path = joinpath(out.images, "$(condition)_notebook_best_fit_overlay.png")
    savefig(best_fit_plot, best_overlay_path)
    println("Saved: $(best_overlay_path)")
    display(best_fit_plot)
else
    best_fit_plot = scatter(
        overlay[!, time_col],
        overlay[!, obs_col];
        alpha = 0.4,
        ms = 3,
        color = :black,
        label = "observed",
        xlabel = "Time (day)",
        ylabel = "Cell count",
        title = "$(condition): fitted model over data",
        legend = :topleft,
    )
    pmask = finite_mask(overlay[!, pred_col])
    plot!(best_fit_plot, overlay[pmask, time_col], overlay[pmask, pred_col]; lw = 3, color = :crimson, label = selected_label)
    best_overlay_path = joinpath(out.images, "$(condition)_notebook_best_fit_overlay.png")
    savefig(best_fit_plot, best_overlay_path)
    println("Source overlay: $(overlay_path)")
    println("Prediction used: $(selected_label)")
    println("Saved: $(best_overlay_path)")
    display(best_fit_plot)
end